# Mapas de tempo de acesso — Curitiba

Um mapa por cenário da pipeline, **todos com a mesma escala de tempo e o mesmo padrão de cores**, para que a comparação visual seja imediata.

O tempo de acesso de uma célula é o PTh: média do tempo de viagem até os *k* POIs mais próximos.


## 1. Bibliotecas e caminhos


In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import colormaps as mpl_colormaps
from matplotlib import colors as mpl_colors
from matplotlib.colors import BoundaryNorm

BASE_DIR = Path.cwd().parent.parent
RESULTS_DIR = BASE_DIR / "results" / "curitiba"
OUTPUT_DIR = Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("raiz:", BASE_DIR)
print("resultados:", RESULTS_DIR)


## 2. Cenários e escala de legendas

A escala e a paleta são definidas **uma vez**, fora do laço de plotagem. É isso que garante que a cor de um mesmo tempo signifique o mesmo em todos os mapas.


In [ ]:
SCENARIOS = [
    "typical",
    "lst",
    "lst_green_composite",
    "lst_demo_k05",
    "lst_green_composite_demo_k05",
]

LABELS = {
    "typical": "Típico",
    "lst": "LST\n(calibrado)",
    "lst_green_composite": "LST + vegetação\n(calibrado)",
    "lst_demo_k05": "LST\n(demonstrativo, k=0,05)",
    "lst_green_composite_demo_k05": "LST + vegetação\n(demonstrativo)",
}

CMAP = "RdYlBu_r"
N_BINS = 12


## 3. Carregamento do tempo de acesso por cenário

Cada matriz origem-destino é reduzida a **um valor por célula**: a média do tempo de viagem aos POIs considerados.


In [ ]:
grid = gpd.read_parquet(RESULTS_DIR / "prepared" / "grid.parquet")
print("células na malha:", len(grid), "| CRS:", grid.crs.to_epsg())

acesso = {}
for cen in SCENARIOS:
    matriz = pd.read_parquet(RESULTS_DIR / "matrices" / f"matrix_{cen}.parquet")
    pth = matriz.groupby("cell_idx")["travel_time"].mean().rename("access_time")
    acesso[cen] = pth
    print(f"{cen:32s} células com valor: {pth.notna().sum():5d} "
          f"| mediana: {pth.median():6.2f} min")


## 4. Escala comum entre os cenários

O mínimo e o máximo são calculados sobre **todos** os cenários juntos.


In [ ]:
todos = pd.concat(acesso.values())
todos = todos[np.isfinite(todos)]

FAIXAS = [0, 10, 15, 30, 50]

CORES = [
    "#1f4e79",   # 0–10   azul forte
    "#7fb3d5",   # 10–15  azul claro
    "#ffe066",   # 15–30  amarelo
    "#ff9f43",   # 30–50  laranja
    "#c0392b",   # 50+    vermelho
]

ROTULOS = [
    "0–10 min",
    "10–15 min",
    "15–30 min",
    "30–50 min",
    "> 50 min",
]

CMAP_MAPA = mpl_colors.ListedColormap(CORES, name="acesso")

LIMITES_NORM = np.array(FAIXAS + [np.inf], dtype=float)
NORM = mpl_colors.BoundaryNorm(LIMITES_NORM, ncolors=len(CORES))

## 5. Mapas individuais

Um arquivo por cenário, todos com a mesma legenda. A célula cinza indica ausência de dado: são 30 células (3,1 km²) que não chegam a nenhum POI pela rede viária, e por isso não têm tempo de acesso calculado.


In [ ]:
from matplotlib.patches import Patch


def plotar(cen: str, ax=None, mostrar_legenda: bool = True, tamanho_titulo: int = 24):
    """Desenha o mapa de tempo de acesso de um cenário na escala compartilhada."""
    celulas = grid.merge(
        acesso[cen].rename("access_time"), left_on="cell_idx", right_index=True, how="left"
    )

    proprio = ax is None
    if proprio:
        fig = plt.figure(figsize=(12, 12))
        ax = fig.add_axes([0.02, 0.16, 0.96, 0.78])

    celulas.plot(
        column="access_time",
        cmap=CMAP_MAPA,
        norm=NORM,
        ax=ax,
        edgecolor="none",
        missing_kwds={"color": "#d9d9d9"},
        legend=False,
    )
    ax.set_title(LABELS[cen], fontsize=tamanho_titulo, pad=14)
    ax.set_axis_off()

    if proprio and mostrar_legenda:
        legenda = fig.legend(
            handles=[Patch(facecolor=cor, edgecolor="#555555", label=rotulo)
                     for cor, rotulo in zip(CORES, ROTULOS)],
            loc="lower center",
            ncol=len(CORES),
            frameon=False,
            fontsize=17,
            handlelength=1.8,
            handleheight=1.4,
            columnspacing=1.8,
            bbox_to_anchor=(0.5, 0.04),
            title="Tempo de acesso (min)",
            title_fontsize=18,
        )
        legenda.get_title().set_fontweight("bold")

    if proprio:
        return fig, ax
    return ax


In [ ]:
for cen in SCENARIOS:
    fig, _ = plotar(cen)
    destino = OUTPUT_DIR / f"curitiba_access_time_{cen}.png"
    fig.savefig(destino, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("salvo:", destino.name)


## 6. Painel comparativo

Os cinco mapas lado a lado numa única escala. É a figura que mostra, de relance, o quanto cada cenário desloca a distribuição do tempo de acesso.


In [ ]:
from matplotlib.patches import Patch

# A mancha urbana é alta e estreita (21,1 km × 33,3 km). Os mapas vão em 3
# colunas, então cada eixo precisa ser mais alto que largo.
LARGURA_EIXO = 5.2
ALTURA_EIXO = LARGURA_EIXO * (33.3 / 21.1)

fig = plt.figure(figsize=(3 * LARGURA_EIXO + 0.4, 2 * ALTURA_EIXO + 3.0))
grade = fig.add_gridspec(
    2, 3, left=0.005, right=0.995, top=0.900, bottom=0.150,
    hspace=0.12, wspace=0.005,
)

for i, cen in enumerate(SCENARIOS):
    ax = fig.add_subplot(grade[i // 3, i % 3])
    plotar(cen, ax=ax, mostrar_legenda=False, tamanho_titulo=21)
    ax.set_aspect("equal")
    ax.set_anchor("C")

fig.add_subplot(grade[1, 2]).set_axis_off()

legenda = fig.legend(
    handles=[Patch(facecolor=cor, edgecolor="#555555", label=rotulo)
             for cor, rotulo in zip(CORES, ROTULOS)],
    loc="lower center",
    ncol=len(CORES),
    frameon=False,
    fontsize=19,
    handlelength=1.8,
    handleheight=1.4,
    columnspacing=2.4,
    bbox_to_anchor=(0.5, 0.020),
    title="Tempo de acesso (min)",
    title_fontsize=20,
)
legenda.get_title().set_fontweight("bold")

fig.suptitle("Tempo de acesso por cenário — Curitiba", fontsize=28, y=0.975)
fig.savefig(OUTPUT_DIR / "curitiba_access_time_painel.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print("salvo: curitiba_access_time_painel.png")
